In [1]:
# @title Idioma base
language = 'pt'  # Ao executar esta célula, assumimos que todo o fluxo será em português.

Bloco 1 🎤 Gravação de Áudio Com Python (e Uma Pitada de JavaScript)

In [2]:
# @title
from IPython.display import Javascript, display, Audio
from base64 import b64decode

JS_RECORD = r"""
async function recordOnceInteractive() {
  // Evita múltiplas UIs/event listeners
  if (!window.__rec_ui) {
    const wrap = document.createElement('div');
    wrap.style.cssText = 'margin:10px 0;padding:10px;border:1px solid #444;border-radius:8px;display:inline-block;';
    const title = document.createElement('div');
    title.textContent = 'Gravação por voz (Start/Stop)';
    title.style.cssText = 'font-weight:600;margin-bottom:6px;';
    const btnStart = document.createElement('button');
    const btnStop  = document.createElement('button');
    btnStart.textContent = '🎙️ Iniciar';
    btnStop.textContent  = '⏹️ Parar';
    btnStart.style.cssText = 'margin-right:8px;';
    btnStop.disabled = true;
    wrap.appendChild(title);
    wrap.appendChild(btnStart);
    wrap.appendChild(btnStop);
    document.body.appendChild(wrap);
    window.__rec_ui = {wrap, btnStart, btnStop};
  }

  const {btnStart, btnStop} = window.__rec_ui;

  // Se já houver gravador/stream ativos, encerra com segurança
  if (window.__rec_stream) {
    try { window.__rec_stream.getTracks().forEach(t => t.stop()); } catch(e) {}
    window.__rec_stream = null;
  }
  window.__rec_chunks = [];

  const b2text = blob => new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = e => resolve(e.srcElement.result);
    reader.readAsDataURL(blob);
  });

  // Garante que o navegador pediu permissão antes
  try {
    await navigator.mediaDevices.getUserMedia({audio:true});
  } catch (e) {
    return JSON.stringify({ ok:false, error: 'Permissão de microfone negada: ' + e.message });
  }

  // Configura eventos de clique (uma única vez)
  if (!btnStart.__bound) {
    btnStart.__bound = true;
    btnStart.addEventListener('click', async () => {
      try {
        window.__rec_stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        window.__rec_recorder = new MediaRecorder(window.__rec_stream);
        window.__rec_chunks = [];
        window.__rec_recorder.ondataavailable = e => window.__rec_chunks.push(e.data);
        window.__rec_recorder.start();
        btnStart.disabled = true;
        btnStop.disabled = false;
      } catch (e) {
        alert('Erro ao iniciar gravação: ' + e.message);
      }
    });
  }

  return await new Promise(resolve => {
    if (!btnStop.__bound) {
      btnStop.__bound = true;
      btnStop.addEventListener('click', async () => {
        try {
          const rec = window.__rec_recorder;
          if (rec && rec.state !== 'inactive') {
            rec.onstop = async () => {
              try {
                const blob = new Blob(window.__rec_chunks, { type: 'audio/webm' });
                const dataUrl = await b2text(blob);
                try { window.__rec_stream.getTracks().forEach(t => t.stop()); } catch(e){}
                window.__rec_stream = null;
                btnStart.disabled = false;
                btnStop.disabled  = true;
                resolve(JSON.stringify({ ok:true, dataUrl }));
              } catch (e) {
                resolve(JSON.stringify({ ok:false, error:'Falha ao converter áudio: '+e.message }));
              }
            };
            rec.stop();
          } else {
            resolve(JSON.stringify({ ok:false, error:'Gravação não estava ativa.' }));
          }
        } catch (e) {
          resolve(JSON.stringify({ ok:false, error:'Erro ao parar: '+e.message }));
        }
      });
    }
    // Instrução visual
    const tip = document.createElement('div');
    tip.textContent = 'Clique em "🎙️ Iniciar" para começar e "⏹️ Parar" para finalizar.';
    tip.style.cssText = 'margin-top:6px;color:#aaa;';
    if (!window.__rec_tip_added) {
      window.__rec_ui.wrap.appendChild(tip);
      window.__rec_tip_added = true;
    }
  });
}
"""

# Injeta JS
display(Javascript(JS_RECORD))

# Executa o fluxo interativo JS e aguarda até você clicar em "Parar"
from google.colab import output
result_json = output.eval_js("recordOnceInteractive()")

import json, os
res = json.loads(result_json)
if not res.get("ok"):
    raise RuntimeError("Falha na gravação: " + str(res.get("error")))

# Converte e salva o arquivo WAV (a partir do dataURL em Base64)
data_url = res["dataUrl"]
audio_b64 = data_url.split(",")[1]
audio = b64decode(audio_b64)

fname = "request_audio.webm"
with open(fname, "wb") as f:
    f.write(audio)

# Converte WEBM -> WAV (Whisper lida com vários formatos, mas deixo WAV por consistência)
# OBS: o Colab tem ffmpeg; a conversão é rápida.
wav_path = "request_audio.wav"
os.system(f'ffmpeg -y -i "{fname}" -ac 1 -ar 16000 -c:a pcm_s16le "{wav_path}" >/dev/null 2>&1')

print("💾 Áudio salvo em:", wav_path)
display(Audio(wav_path, autoplay=False))

<IPython.core.display.Javascript object>

💾 Áudio salvo em: request_audio.wav


Bloco 2 🧠 Reconhecimento de Fala com Whisper (OpenAI)

In [3]:
# @title
import torch, whisper

MODEL_GPU = "small"   # bom equilíbrio em GPU (pode usar "medium" para mais acurácia)
MODEL_CPU = "base"    # fallback caso esteja em CPU
FORCED_LANGUAGE = "pt"  # acelera a transcrição em PT-BR

use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"
model_name = MODEL_GPU if use_cuda else MODEL_CPU
print(f"Dispositivo: {device} | Modelo: {model_name}")

model = whisper.load_model(model_name, device=device)
decode_kwargs = {"fp16": False} if device == "cpu" else {}

audio_path = "request_audio.wav"
result = model.transcribe(audio_path, language=FORCED_LANGUAGE, **decode_kwargs)

transcription_text = result["text"]
transcription_lang = result.get("language", FORCED_LANGUAGE)

print("✅ Transcrição:", transcription_text)
print("🌍 Idioma detectado:", transcription_lang)

Dispositivo: cuda | Modelo: small
✅ Transcrição:  Agenar uma reunião com a equipe do novo ataque a rejo na terça-feira da próxima semana.
🌍 Idioma detectado: pt


Bloco 3 🗝️ Reconhecimento de Fala com Whisper (OpenAI)

In [4]:
# @title
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Digite sua OPENAI_API_KEY: ")

OPENAI_CHAT_MODEL = "gpt-4o-mini"
print("API Key configurada somente em tempo de execução.")


Digite sua OPENAI_API_KEY: ··········
API Key configurada somente em tempo de execução.


Bloco 4 🧠 NLU resiliente (API + fallback local)

In [12]:
# Bloco 4 — NLU resiliente (API + fallback local) — com normalização forte de título
import os, json, re, unicodedata, time
from datetime import datetime
import dateparser
from dateparser.search import search_dates
import pytz

# =========================
# Configs
# =========================
TZ = "America/Fortaleza"
OPENAI_CHAT_MODEL = "gpt-4o-mini"

# =========================
# Utilitários de normalização
# =========================
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalize_title_pt(title: str) -> str:
    """
    Remove restos de dia/semana/preposições/pontuação e devolve um título limpo.
    Se ficar vazio/curto, retorna 'Compromisso'.
    """
    if not title:
        return "Compromisso"

    # Trabalhamos numa cópia sem acentos para remoção, mas devolvemos capitalizado simples.
    raw = title.strip()
    low_noacc = _strip_accents(raw.lower())

    # 1) Remover padrões de dias da semana (com ou sem '-feira') + sufixos de semana
    #    Ex.: "terça-feira da próxima semana", "sexta feira", "quarta da semana que vem"
    patterns = [
        r"\b(segunda|terca|terça|quarta|quinta|sexta)(?:-|\s)?feira\b(?:\s+(?:da|de|do)\s+(?:proxima\s+semana|semana\s+que\s+vem))?",
        r"\b(domingo|sabado|sábado)\b(?:\s+(?:da|de|do)\s+(?:proxima\s+semana|semana\s+que\s+vem))?",
        r"\b(da|de|do)\s+proxima\s+semana\b",
        r"\b(da|de|do)\s+semana\s+que\s+vem\b",
        r"\b(amanha|amanhã|hoje|depois\s+de\s+amanha|depois\s+do\s+almoco)\b",
    ]
    temp = low_noacc
    for p in patterns:
        temp = re.sub(p, "", temp, flags=re.IGNORECASE)

    # 2) Remover preposições soltas comuns no início/fim e pontuação residual
    temp = re.sub(r"^\b(na|no|em|para|pra)\b\s*", "", temp, flags=re.IGNORECASE)
    temp = re.sub(r"\s*\b(na|no|em|para|pra)\b$", "", temp, flags=re.IGNORECASE)
    temp = temp.strip(" .,:;-–—\t\n").strip()

    # 3) Se sobrar algo muito curto ou lixo, padroniza
    if len(temp) < 3 or temp in {"feira", "-feira"}:
        return "Compromisso"

    # 4) Capitaliza (simples)
    return temp[0].upper() + temp[1:]

# =========================
# Utilitários de data/hora
# =========================
def to_iso_local(texto, base_tz=TZ):
    tz = pytz.timezone(base_tz)
    settings = {
        "PREFER_DATES_FROM": "future",
        "TIMEZONE": base_tz,
        "TO_TIMEZONE": base_tz,
        "RETURN_AS_TIMEZONE_AWARE": True,
        "RELATIVE_BASE": datetime.now(tz),
    }
    dt = dateparser.parse(texto, settings=settings, languages=["pt"])
    if not dt:
        return None
    if not dt.tzinfo:
        dt = tz.localize(dt)
    return dt.astimezone(tz).isoformat(timespec="minutes")

def first_datetime_iso(texto, base_tz=TZ):
    tz = pytz.timezone(base_tz)
    settings = {
        "PREFER_DATES_FROM": "future",
        "TIMEZONE": base_tz,
        "TO_TIMEZONE": base_tz,
        "RETURN_AS_TIMEZONE_AWARE": True,
        "RELATIVE_BASE": datetime.now(tz),
    }
    found = search_dates(texto, settings=settings, languages=["pt"])
    if not found:
        return None
    dt = found[0][1]
    if not dt.tzinfo:
        dt = tz.localize(dt)
    return dt.astimezone(tz).isoformat(timespec="minutes")

# =========================
# Extração de título para eventos
# =========================
def extract_event_title(texto: str) -> str:
    """
    Regras:
      1) Se houver 'reunião com <alguém>', usa 'Reunião com <alguém>'.
      2) Caso contrário, pega o trecho após o gatilho (reunião/encontro/agendar/evento)
         até antes de tokens de data/hora/relativos, e normaliza.
      3) Fallback: 'Compromisso'.
    """
    t = texto.strip()

    # 1) 'reunião com <nome>'
    m_com = re.search(r"reuni[aã]o\s+com\s+([A-Za-zÀ-ÿ'´`^~\s]{2,60})", t, flags=re.IGNORECASE)
    if m_com:
        nome = m_com.group(1).strip()
        # corta sufixos de data/hora que eventualmente grudam
        nome = re.split(r"(?:\s+amanh[ãa]|(?:\s+às|\s+as)\s+\d|(?:\s+na|\s+no|\s+em)\s+)", nome)[0].strip()
        return normalize_title_pt(f"Reunião com {nome}")

    # 2) trecho após gatilho
    gatilho = re.search(r"(reuni[aã]o|encontro|agendar|evento)\b", t, flags=re.IGNORECASE)
    if gatilho:
        after = t[gatilho.end():].strip()
        # corta em tokens típicos de tempo/lugar
        cuts = [
            " amanhã", " hoje", " às ", " as ", " a ",
            " daqui a ", " de ", " no ", " na ", " em ",
            " segunda", " terça", " terca", " quarta", " quinta", " sexta",
            " sábado", " sabado", " domingo",
            " semana que vem", " próxima semana", " proxima semana"
        ]
        end_idx = len(after)
        low = after.lower()
        for c in cuts:
            idx = low.find(c)
            if idx != -1:
                end_idx = min(end_idx, idx)
        base = after[:end_idx].strip()
        if base:
            return normalize_title_pt(base)

    # 3) fallback
    return "Compromisso"

# =========================
# Heurística OFFLINE (sem API)
# =========================
def nlu_offline(texto: str):
    tlow_noacc = _strip_accents(texto.strip().lower())

    # GET SUMMARY
    if re.search(r"\b(resumo|o que eu tenho|minha agenda)\b", tlow_noacc):
        return {
            "intent": "get_summary",
            "language": "pt-BR",
            "entities": {"title": None, "datetime": None, "end_datetime": None, "note_text": None},
            "natural_response": "Beleza, vou conferir e já te digo."
        }

    # CREATE NOTE
    if re.search(r"\b(anote|nota|anotar|guardar)\b", tlow_noacc):
        m = re.search(r"(anote|nota|anotar|guardar)\s*(.+)", texto, flags=re.IGNORECASE)
        note_text = (m.group(2).strip() if m else texto).strip()
        return {
            "intent": "create_note",
            "language": "pt-BR",
            "entities": {"title": None, "datetime": None, "end_datetime": None, "note_text": note_text},
            "natural_response": "Anotado! Quer que eu categorize essa nota?"
        }

    # CREATE EVENT
    if re.search(r"\b(reuniao|reuni[aã]o|encontro|agendar|evento)\b", tlow_noacc):
        title = extract_event_title(texto)
        dt = first_datetime_iso(texto, base_tz=TZ)
        natural = "Evento registrado! Quer definir local ou duração?"
        if not dt:
            natural = "Para quando é esse evento? Pode dizer a data e o horário."
        return {
            "intent": "create_event",
            "language": "pt-BR",
            "entities": {
                "title": title, "datetime": dt, "end_datetime": None, "note_text": None
            },
            "natural_response": natural
        }

    # CREATE TASK
    if re.search(r"\b(tarefa|lembre(-)?me|lembrar de|adicionar tarefa|criar tarefa)\b", tlow_noacc):
        desc = re.sub(r"\b(tarefa|lembre(-)?me|lembrar de|adicionar tarefa|criar tarefa)\b", "", texto, flags=re.IGNORECASE).strip()
        title = normalize_title_pt(desc) if desc else "Tarefa"
        dt = first_datetime_iso(texto, base_tz=TZ)
        natural = "Perfeito — tarefa registrada." if dt else "Quando você quer que eu te lembre dessa tarefa?"
        return {
            "intent": "create_task",
            "language": "pt-BR",
            "entities": {
                "title": title, "datetime": dt, "end_datetime": None, "note_text": None
            },
            "natural_response": natural
        }

    # SMALLTALK / FALLBACK
    return {
        "intent": "smalltalk",
        "language": "pt-BR",
        "entities": {"title": None, "datetime": None, "end_datetime": None, "note_text": None},
        "natural_response": "Certo! Me diga se é uma tarefa, evento, nota ou deseja um resumo."
    }

# =========================
# NLU resiliente (API + backoff + normalização de título SEMPRE)
# =========================
def interpretar_texto_resiliente(texto: str, timezone=TZ, locale="pt-BR",
                                 max_retries=4, base_delay=1.5, usar_api=True):
    """
    1) Tenta API OpenAI (com retry/backoff).
    2) Se falhar (429/quota etc.), usa heurística offline.
    3) Em ambos os casos, normaliza entities.title antes de retornar.
    """
    if usar_api:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
            SYSTEM_PROMPT = (
                "Você é um NLU para um assistente de produtividade por voz.\n"
                "Saída SEMPRE em JSON com o formato:\n\n"
                "{\n"
                '  "intent": "create_task|create_event|create_note|get_summary|smalltalk",\n'
                '  "language": "pt-BR|en|es|auto",\n'
                '  "entities": {\n'
                '    "title": null,\n'
                '    "datetime": null,\n'
                '    "end_datetime": null,\n'
                '    "note_text": null\n'
                "  },\n"
                '  "natural_response": "Texto curto e educado no idioma de entrada."\n'
                "}\n\n"
                "- Converta tempos relativos para ISO quando possível.\n"
                "- Se faltar informação essencial, deixe null e peça via natural_response.\n"
                "- NÃO inclua nada fora do JSON."
            )
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT + f"\nTimezone:{timezone}\nLocale:{locale}"},
                {"role": "user", "content": texto}
            ]
            for i in range(max_retries):
                try:
                    resp = client.chat.completions.create(
                        model=OPENAI_CHAT_MODEL,
                        messages=msgs,
                        temperature=0.2,
                        response_format={"type": "json_object"}
                    )
                    j = json.loads(resp.choices[0].message.content)
                    # Normaliza SEMPRE o título vindo da API
                    if j.get("entities", {}).get("title") is not None:
                        j["entities"]["title"] = normalize_title_pt(j["entities"]["title"])
                    return j
                except Exception as e:
                    if i < max_retries - 1 and ("429" in str(e) or "quota" in str(e).lower()):
                        wait = base_delay * (2**i)
                        print(f"[NLU/API] 429/quota. Tentativa {i+1}/{max_retries}. Aguardando {wait:.1f}s...")
                        time.sleep(wait)
                        continue
                    print("[NLU/API] Falha, usando heurística offline:", e)
                    break
        except Exception as e:
            print("[NLU] Erro ao iniciar cliente OpenAI — usando heurística offline:", e)

    # Fallback offline
    j = nlu_offline(texto)
    if j.get("entities", {}).get("title") is not None:
        j["entities"]["title"] = normalize_title_pt(j["entities"]["title"])
    return j

# --------- EXECUÇÃO ---------
nlu = interpretar_texto_resiliente(transcription_text, timezone=TZ, locale="pt-BR", usar_api=True)
print("NLU (resiliente — título normalizado):", json.dumps(nlu, ensure_ascii=False, indent=2))

[NLU/API] 429/quota. Tentativa 1/4. Aguardando 1.5s...
[NLU/API] 429/quota. Tentativa 2/4. Aguardando 3.0s...
[NLU/API] 429/quota. Tentativa 3/4. Aguardando 6.0s...
[NLU/API] Falha, usando heurística offline: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
NLU (resiliente — título normalizado): {
  "intent": "create_event",
  "language": "pt-BR",
  "entities": {
    "title": "Reuniao com a equipe do novo ataque a rejo",
    "datetime": "2026-03-17T00:00-03:00",
    "end_datetime": null,
    "note_text": null
  },
  "natural_response": "Evento registrado! Quer definir local ou duração?"
}


Bloco 5 🎧 Executor com follow up por voz (offline)

In [13]:
# === Bloco 5 — Executor com follow-up interativo (Start/Stop), título limpo e horário robusto ===
import os, re, subprocess, unicodedata
from datetime import datetime
import pytz, dateparser
from dateparser.search import search_dates
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import torch, whisper

# =========================
# CONFIGURAÇÕES
# =========================
TZ = "America/Fortaleza"
tz = pytz.timezone(TZ)

# Se True, SEMPRE pergunta o horário (mesmo se o NLU já trouxe hora).
# Se False, pergunta só quando a hora estiver ausente/00:00.
ALWAYS_ASK = False

# =========================
# HELPERS: ÁUDIO / TTS
# =========================
def tts_play(texto: str, lang="pt", fname="prompt_followup.wav", autoplay=True):
    """Fala 'texto' com gTTS e reproduz no notebook."""
    tts = gTTS(text=texto, lang=lang, slow=False)
    tts.save(fname)
    display(Audio(fname, autoplay=autoplay))

# Gravador interativo (Start/Stop). A célula só segue após clicar em "Parar".
JS_REC_INTERACTIVE = r"""
async function recordInteractiveOnce(){
  if (!window.__fui) {
    const wrap = document.createElement('div');
    wrap.style.cssText = 'margin:8px 0;padding:10px;border:1px solid #666;border-radius:8px;display:inline-block;';
    const title = document.createElement('div');
    title.textContent = 'Follow-up de horário (Start/Stop)';
    title.style.cssText = 'font-weight:600;margin-bottom:6px;';
    const tip = document.createElement('div');
    tip.textContent = 'Clique em "🎙️ Iniciar", fale o horário, e depois clique "⏹️ Parar".';
    tip.style.cssText = 'margin-bottom:6px;color:#a0a0a0;';
    const startBtn = document.createElement('button');
    const stopBtn  = document.createElement('button');
    startBtn.textContent = '🎙️ Iniciar';
    stopBtn.textContent  = '⏹️ Parar';
    startBtn.style.marginRight = '8px';
    stopBtn.disabled = true;
    wrap.appendChild(title); wrap.appendChild(tip);
    wrap.appendChild(startBtn); wrap.appendChild(stopBtn);
    document.body.appendChild(wrap);
    window.__fui = {wrap, startBtn, stopBtn};
  }
  const {startBtn, stopBtn} = window.__fui;
  let stream, recorder, chunks = [];

  const b2text = blob => new Promise(res => {
    const reader = new FileReader();
    reader.onloadend = e => res(e.srcElement.result);
    reader.readAsDataURL(blob);
  });

  if (!startBtn.__bound) {
    startBtn.__bound = true;
    startBtn.addEventListener('click', async () => {
      try {
        stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        recorder = new MediaRecorder(stream);
        chunks = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();
        startBtn.disabled = true; stopBtn.disabled = false;
      } catch(e) { alert('Erro ao iniciar: '+e.message); }
    });
  }

  return await new Promise(resolve => {
    if (!stopBtn.__bound) {
      stopBtn.__bound = true;
      stopBtn.addEventListener('click', async () => {
        try {
          if (recorder && recorder.state !== 'inactive') {
            recorder.onstop = async () => {
              const blob = new Blob(chunks, {type:'audio/webm'});
              const dataUrl = await b2text(blob);
              if (stream) stream.getTracks().forEach(t=>t.stop());
              startBtn.disabled = false; stopBtn.disabled = true;
              resolve(dataUrl);
            };
            recorder.stop();
          } else {
            resolve(null);
          }
        } catch(e) { resolve(null); }
      });
    }
  });
}
"""

def record_js_to_wav_interactive(out_wav="followup.wav"):
    """Abre UI Start/Stop, espera você clicar em Parar e converte WEBM->WAV 16 kHz mono."""
    display(Javascript(JS_REC_INTERACTIVE))
    data_url = output.eval_js('recordInteractiveOnce()')
    if not data_url:
        raise RuntimeError("Falha ao capturar áudio do follow-up.")
    audio_b64 = data_url.split(",")[1]
    webm_file = "followup.webm"
    with open(webm_file, "wb") as f:
        f.write(b64decode(audio_b64))
    subprocess.run(
        ["ffmpeg", "-y", "-i", webm_file, "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", out_wav],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    return out_wav

# =========================
# HELPERS: STT LOCAL (Whisper)
# =========================
def transcribe_local(wav_path: str, forced_lang="pt"):
    """Transcreve com Whisper local (usa GPU se disponível)."""
    MODEL_GPU = "small"
    MODEL_CPU = "base"
    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"
    model_name = MODEL_GPU if use_cuda else MODEL_CPU
    model = whisper.load_model(model_name, device=device)
    decode_kwargs = {"fp16": False} if device == "cpu" else {}
    result = model.transcribe(wav_path, language=forced_lang, **decode_kwargs)
    return result["text"].strip()

# =========================
# HELPERS: TÍTULO / TEMPO / DATETIME
# =========================
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def normalize_title_pt(title: str) -> str:
    """Limpa restos de dia/semana/preposições e padroniza título."""
    if not title:
        return "Compromisso"
    t = title.strip()
    low_noacc = _strip_accents(t.lower())

    lixo = [
        r"\b(segunda|terca|terça|quarta|quinta|sexta)(-|\s)?feira\b",
        r"\b(domingo|sabado|sábado)\b",
        r"\b(da|de|do)\s+proxima\s+semana\b",
        r"\b(da|de|do)\s+semana\s+que\s+vem\b",
        r"\b(amanha|amanhã|hoje|depois\s+de\s+amanha|depois\s+do\s+almoco)\b",
        r"\b(na|no|em|para|pra)\b",
    ]
    # Aplica remoções sobre versão sem acento…
    temp = low_noacc
    for p in lixo:
        temp = re.sub(p, "", temp, flags=re.IGNORECASE)
    # …mas retorna com capitalização simples
    temp = temp.strip(" .,:;-–—\t\n").strip()
    if len(temp) < 3 or temp in {"feira", "-feira"}:
        return "Compromisso"
    return temp[0].upper() + temp[1:]

def parse_local_iso(texto: str, base_tz=TZ):
    """Converte string para datetime timezone-aware no fuso local."""
    dt = dateparser.parse(texto)
    if not dt:
        return None
    tz_ = pytz.timezone(base_tz)
    if dt.tzinfo is None:
        dt = tz_.localize(dt)
    else:
        dt = dt.astimezone(tz_)
    return dt

def is_time_missing(starts_iso: str) -> bool:
    """True se hora está 00:00 (sem horário)."""
    dt = parse_local_iso(starts_iso, TZ)
    if not dt:
        return True
    return dt.hour == 0 and dt.minute == 0

def parse_time_phrase_pt(time_phrase: str, base_dt: datetime, tz=pytz.timezone(TZ)) -> datetime | None:
    """
    Entende: 'quatro da tarde'->16:00, '16 horas da tarde'->16:00,
             'três e meia'->03:30 (ou 15:30 com 'da tarde'), 'quinze e trinta'->15:30,
             'meio-dia'->12:00, 'meia-noite'->00:00.
    Retorna datetime no MESMO DIA de base_dt (timezone-aware).
    """
    raw = time_phrase.strip()
    low = raw.lower()
    low_noacc = _strip_accents(low)

    # Casos especiais
    if "meio-dia" in low or "meio dia" in low_noacc:
        hh, mm = 12, 0
    elif "meia-noite" in low or "meia noite" in low_noacc:
        hh, mm = 0, 0
    else:
        meia = False
        if " e meia" in low or " e meia" in low_noacc:
            meia = True
            low = low.replace(" e meia", "")
            low_noacc = low_noacc.replace(" e meia", "")

        # Numérico: 16, 16:30, 16h30, 16h, 16 30, 4, 4:30 etc.
        m = re.search(r"\b([01]?\d|2[0-3])(?:[:h\. ]?([0-5]\d))?\b", low_noacc)
        if m:
            hh = int(m.group(1))
            mm = int(m.group(2)) if m.group(2) else (30 if meia else 0)
        else:
            # Palavras
            WORD_TO_H = {
                "uma":1,"duas":2,"dois":2,"tres":3,"três":3,"quatro":4,"cinco":5,"seis":6,
                "sete":7,"oito":8,"nove":9,"dez":10,"onze":11,"doze":12
            }
            hh, mm = None, (30 if meia else 0)
            for w,hv in WORD_TO_H.items():
                if re.search(rf"\b{_strip_accents(w)}\b", low_noacc):
                    hh = hv; break
            if hh is None:
                return None

        # AM/PM pelo contexto
        has_tarde = re.search(r"\b(da|de)\s+tarde\b", low_noacc)
        has_noite = re.search(r"\b(da|de)\s+noite\b", low_noacc)
        has_manha = re.search(r"\b(da|de)\s+manha\b", low_noacc)

        # Soma 12 apenas se 1..11 e citou tarde/noite
        if (has_tarde or has_noite) and 1 <= hh <= 11:
            hh += 12
        # "12 da manhã" -> 00
        if has_manha and hh == 12:
            hh = 0
        # Se já está em 24h (ex.: "16 horas da tarde"), não altera.

        hh = max(0, min(23, hh))
        mm = max(0, min(59, mm))

    # monta no mesmo dia corretamente (pytz: usar localize)
    naive = datetime(base_dt.year, base_dt.month, base_dt.day, hh, mm)
    return tz.localize(naive)

def merge_date_with_time(date_iso: str, time_phrase: str) -> str | None:
    """
    Prioridade: (1) parser manual PT-BR -> (2) search_dates -> (3) dateparser.parse
    Retorna ISO-8601 no fuso TZ.
    """
    # base_dt ISO -> timezone-aware corretamente
    try:
        base_dt = datetime.fromisoformat(date_iso)
        tz_ = pytz.timezone(TZ)
        if base_dt.tzinfo is None:
            base_dt = tz_.localize(base_dt)
        else:
            base_dt = base_dt.astimezone(tz_)
    except Exception:
        base_dt = parse_local_iso(date_iso, TZ)
        if not base_dt:
            return None

    # (1) manual primeiro
    manual = parse_time_phrase_pt(time_phrase, base_dt, pytz.timezone(TZ))
    if manual:
        print(f"[merge] manual → {manual.isoformat(timespec='minutes')}")
        return manual.isoformat(timespec="minutes")

    # (2) search_dates
    settings = {
        "PREFER_DATES_FROM": "future",
        "TIMEZONE": TZ, "TO_TIMEZONE": TZ,
        "RETURN_AS_TIMEZONE_AWARE": True,
        "RELATIVE_BASE": base_dt,
    }
    found = search_dates(time_phrase, settings=settings, languages=["pt"])
    if found:
        cand = found[0][1].astimezone(pytz.timezone(TZ))
        naive = datetime(base_dt.year, base_dt.month, base_dt.day, cand.hour, cand.minute)
        merged = pytz.timezone(TZ).localize(naive)
        print(f"[merge] search_dates → {merged.isoformat(timespec='minutes')}")
        return merged.isoformat(timespec="minutes")

    # (3) dateparser.parse
    dp = dateparser.parse(time_phrase, settings=settings, languages=["pt"])
    if dp:
        dp = dp.astimezone(pytz.timezone(TZ))
        naive = datetime(base_dt.year, base_dt.month, base_dt.day, dp.hour, dp.minute)
        merged = pytz.timezone(TZ).localize(naive)
        print(f"[merge] dateparser.parse → {merged.isoformat(timespec='minutes')}")
        return merged.isoformat(timespec="minutes")

    print("[merge] nenhum horário reconhecido no follow-up.")
    return None

# =========================
# "BANCO" EM MEMÓRIA (MVP)
# =========================
try:
    TASKS
except NameError:
    TASKS, EVENTS, NOTES = [], [], []

# =========================
# EXECUTOR COM FOLLOW-UP
# =========================
def executar_intent_com_followup(nlu: dict):
    """
    Executa intents. Para 'create_event' sem horário (ou se ALWAYS_ASK=True),
    pergunta por voz, grava (Start/Stop), transcreve (Whisper local) e completa.
    """
    intent = nlu.get("intent")
    ent = nlu.get("entities", {}) or {}
    resposta = nlu.get("natural_response") or "Ok."

    # ---- EVENTO ----
    if intent == "create_event" and ent.get("title") and ent.get("datetime"):
        starts_iso = ent["datetime"]

        if ALWAYS_ASK or is_time_missing(starts_iso):
            dt_local = parse_local_iso(starts_iso).astimezone(tz)
            data_br = dt_local.strftime("%d/%m/%Y")
            pergunta = (
                f"Qual horário para o evento em {data_br}? "
                "Pode dizer, por exemplo: 'quinze e trinta', 'três e meia da tarde', 'nove da manhã'. "
                "Clique Iniciar, fale, e depois clique Parar."
            )
            tts_play(pergunta)

            # Grava com Start/Stop (o código só continua após você clicar Parar)
            wav = record_js_to_wav_interactive("followup.wav")
            follow_text = transcribe_local(wav, forced_lang="pt")
            print("🗣️ Follow-up (transcrito):", follow_text)

            merged_iso = merge_date_with_time(starts_iso, follow_text)
            if merged_iso:
                ent["datetime"] = merged_iso
                hr = datetime.fromisoformat(merged_iso).strftime("%H:%M")
                resposta = f"Perfeito. Agendei {normalize_title_pt(ent['title'])} para {data_br} às {hr}."
            else:
                resposta = f"Tudo bem. Mantive {normalize_title_pt(ent['title'])} sem horário definido no dia {data_br}."

        # Registra evento (título normalizado)
        title_norm = normalize_title_pt(ent["title"])
        EVENTS.append({
            "title": title_norm,
            "starts_at": ent["datetime"],
            "ends_at": ent.get("end_datetime")
        })
        # Confirma por voz
        tts = gTTS(text=resposta, lang="pt", slow=False)
        tts.save("response_audio.wav")
        display(Audio("response_audio.wav", autoplay=True))
        return resposta

    # ---- TAREFA ----
    if intent == "create_task" and ent.get("title"):
        TASKS.append({"title": normalize_title_pt(ent["title"]), "due": ent.get("datetime")})
        resposta = "Perfeito — tarefa registrada."
        tts = gTTS(text=resposta, lang="pt", slow=False)
        tts.save("response_audio.wav")
        display(Audio("response_audio.wav", autoplay=True))
        return resposta

    # ---- NOTA ----
    if intent == "create_note" and ent.get("note_text"):
        NOTES.append({"text": ent["note_text"], "created_at": datetime.utcnow().isoformat()})
        resposta = "Anotado!"
        tts = gTTS(text=resposta, lang="pt", slow=False)
        tts.save("response_audio.wav")
        display(Audio("response_audio.wav", autoplay=True))
        return resposta

    # ---- RESUMO ----
    if intent == "get_summary":
        resposta = f"Você tem {len(EVENTS)} compromissos e {len(TASKS)} tarefas pendentes."
        tts = gTTS(text=resposta, lang="pt", slow=False)
        tts.save("response_audio.wav")
        display(Audio("response_audio.wav", autoplay=True))
        return resposta

    # ---- SMALLTALK / FALLBACK ----
    tts = gTTS(text=resposta, lang="pt", slow=False)
    tts.save("response_audio.wav")
    display(Audio("response_audio.wav", autoplay=True))
    return resposta

# =========================
# RODAR O EXECUTOR (usa o 'nlu' da célula anterior)
# =========================
try:
    nlu
except NameError:
    raise RuntimeError("Variável 'nlu' não encontrada. Rode o bloco de NLU antes do Bloco 6.")

natural_response = executar_intent_com_followup(nlu)

print("RESPOSTA:", natural_response)
print("TASKS:", TASKS)
print("EVENTS:", EVENTS)
print("NOTES:", NOTES)

<IPython.core.display.Javascript object>

🗣️ Follow-up (transcrito): Agendar para às 16 horas.
[merge] manual → 2026-03-17T16:00-03:00


RESPOSTA: Perfeito. Agendei Reuniao com a equipe do novo ataque a rejo para 17/03/2026 às 16:00.
TASKS: []
EVENTS: [{'title': '-feira da próxima semana.', 'starts_at': '2026-03-17T08:00-02:34', 'ends_at': None}, {'title': 'Compromisso', 'starts_at': '2026-03-17T16:00-03:00', 'ends_at': None}, {'title': 'Reuniao com a equipe do novo ataque a rejo', 'starts_at': '2026-03-17T16:00-03:00', 'ends_at': None}]
NOTES: []
